# Nested logit (NL)

Estimate an NL model with TRAIN and SM grouped in a public-transport nest.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import pandas as pd
import torch

from torchdcm import (
    AlternativeScale,
    Beta,
    ChoiceDataset,
    ChoiceLatentEffect,
    ContinuousIndicator,
    CovariateScale,
    CovariateScaledMultinomialLogit,
    CrossNest,
    CrossNestedLogit,
    ErrorComponent,
    ErrorComponentsLogit,
    HybridChoiceModel,
    LatentClassLogit,
    LatentVariable,
    MixedLogit,
    MultinomialLogit,
    Nest,
    NestedLogit,
    OrderedChoiceDataset,
    OrderedLogit,
    OrderedProbit,
    RandomCoefficient,
    ScaledMultinomialLogit,
    UtilitySpec,
    WTPCoefficient,
    WTPMixedLogit,
)
from torchdcm.datasets import make_swissmetro_like

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


def show_result(result):
    print(f"Final log likelihood: {result.loglike:.6f}")
    print(f"AIC: {result.aic:.3f}; BIC: {result.bic:.3f}")
    return pd.DataFrame(
        {
            "estimate": result.values,
            "std. error": result.bse,
            "z": result.zvalues,
            "p-value": result.pvalues,
        },
        index=pd.Index(result.param_names, name="parameter"),
    ).round(6)


TorchDCM example running on cuda


In [2]:
def make_choice_data(n_obs=180, seed=7, *, observation_variables=None):
    frame = make_swissmetro_like(n_obs=n_obs, seed=seed)
    data = ChoiceDataset.from_wide(
        frame,
        alternatives=["TRAIN", "SM", "CAR"],
        choice="choice",
        variables={
            "time": {
                "TRAIN": "time_train",
                "SM": "time_sm",
                "CAR": "time_car",
            },
            "cost": {
                "TRAIN": "cost_train",
                "SM": "cost_sm",
                "CAR": "cost_car",
            },
        },
        obs_variables=observation_variables,
        availability={
            "TRAIN": "avail_train",
            "SM": "avail_sm",
            "CAR": "avail_car",
        },
        individual_id="person_id",
    )
    return frame, data


def make_utility_spec(suffix=""):
    tag = f"_{suffix}" if suffix else ""
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    return spec

rng = np.random.default_rng(120)
frame = make_swissmetro_like(n_obs=600, seed=12)
frame[["avail_train", "avail_sm", "avail_car"]] = 1
initial_data = ChoiceDataset.from_wide(
    frame,
    alternatives=["TRAIN", "SM", "CAR"],
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)

dgp_spec = UtilitySpec()
dgp_spec.utility(
    "TRAIN",
    Beta("ASC_TRAIN_DGP", init=0.30)
    + Beta("B_TIME_DGP", init=-0.03) * "time"
    + Beta("B_COST_DGP", init=-0.05) * "cost",
)
dgp_spec.utility(
    "SM",
    Beta("B_TIME_DGP", init=-0.03) * "time"
    + Beta("B_COST_DGP", init=-0.05) * "cost",
)
dgp_spec.utility(
    "CAR",
    Beta("ASC_CAR_DGP", init=0.20)
    + Beta("B_TIME_DGP", init=-0.03) * "time"
    + Beta("B_COST_DGP", init=-0.05) * "cost",
)
dgp = NestedLogit(
    dgp_spec,
    {
        "PUBLIC": Nest(["TRAIN", "SM"], init=0.65),
        "PRIVATE": Nest(["CAR"], init=1.00, fixed=True),
    },
)
true_params = torch.tensor([0.30, -0.03, -0.05, 0.20, 0.65])
probabilities = (
    dgp.predict_proba(initial_data, true_params)
    .reshape(-1, 3)
    .detach()
    .cpu()
    .numpy()
)
alternatives = np.array(["TRAIN", "SM", "CAR"])
frame["choice"] = [
    rng.choice(alternatives, p=row) for row in probabilities
]
data = ChoiceDataset.from_wide(
    frame,
    alternatives=alternatives.tolist(),
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)
spec = make_utility_spec()


In [3]:
nests = {
    "PUBLIC": Nest(["TRAIN", "SM"], init=0.75),
    "PRIVATE": Nest(["CAR"], init=1.00, fixed=True),
}
model = NestedLogit(spec, nests, device=device, max_iter=80)
result = model.fit(data)
show_result(result)


Final log likelihood: -583.829173
AIC: 1177.658; BIC: 1199.643


,estimate,std. error,z,p-value
parameter,,,,
ASC_TRAIN,0.381485,0.130040,2.933595,0.003351
B_TIME,-0.024663,0.005451,-4.524131,0.000006
B_COST,-0.039501,0.012292,-3.213673,0.001310
ASC_CAR,-0.207955,0.240341,-0.865251,0.386901
LAMBDA_PUBLIC,0.396821,0.110550,3.589502,0.000331
